# Wav2Lip Lip-Sync + GFPGAN on Colab **T4**

Makes a face speak an audio clip, then sharpens the face with GFPGAN.

### Runs on the Colab T4 only — NOT HuggingFace ZeroGPU.
Clones open-source Wav2Lip and runs on the T4 Colab gives your runtime; never
calls the hosted HF Space, so ZeroGPU is never involved.

**First:** Runtime -> Change runtime type -> **T4 GPU** -> Save, then run cells
top to bottom. Step 0 proves you got a T4.

### Two settings that matter:
- **OUT_HEIGHT** (Step 4): processing resolution. 1408 is good; raise to 2112
  if you get `Face not detected`. Higher = bigger face = better sync + sharper.
- **Step 5 GFPGAN**: restores/sharpens the face after sync. This is what lifts a
  soft, distant face. Leave it on.
The full frame (whole parking lot) is always preserved; only the mouth is
replaced, then the face is enhanced.


## Step 0 - Prove the GPU is a T4 (not ZeroGPU, not CPU)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.'
print('Torch is using:', torch.cuda.get_device_name(0), '- Colab hardware, not HF ZeroGPU.')


## Step 1 - Get Wav2Lip and install deps


In [ ]:
import os
if not os.path.isdir('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
%cd /content/Wav2Lip
!pip install -q -r requirements.txt
!pip install -q batch-face
print('deps installed')


## Step 2 - Download the Wav2Lip checkpoints


In [ ]:
%cd /content/Wav2Lip
import os
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('face_detection/detection/sfd', exist_ok=True)
B='https://github.com/justinjohn0306/Wav2Lip/releases/download/models/'
!wget -q -c $B'wav2lip_gan.pth' -O checkpoints/wav2lip_gan.pth
!wget -q -c $B's3fd.pth'        -O face_detection/detection/sfd/s3fd.pth
print('checkpoints:', os.listdir('checkpoints'))


## Step 3 - Upload your inputs
**FACE** = your clip (the WIDE `joker-walk-14b` clip is fine — you keep the
whole shot). **AUDIO** = the voice line (wav/mp3).


In [ ]:
from google.colab import files
print('Upload the FACE (mp4 / png / jpg):')
FACE = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('Upload the AUDIO (wav / mp3):')
AUDIO = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('FACE :', FACE); print('AUDIO:', AUDIO)


## Step 4 - Run Wav2Lip on the T4
**OUT_HEIGHT** is the key knob: 1408 to start, raise to 2112 if you see
`Face not detected`. Leave `CROP = None` for a wide shot. Prints the full
error if inference fails.


In [ ]:
OUT_HEIGHT = 1408      # raise to 2112 if 'Face not detected'
PADS = [0, 10, 0, 0]   # top bottom left right
BOX  = None            # last resort: [top,bottom,left,right] on the OUT_HEIGHT frame
CROP = None            # leave None for a wide shot

cmd = ['python','inference.py','--checkpoint_path','checkpoints/wav2lip_gan.pth',
       '--face', FACE, '--audio', AUDIO, '--outfile','/content/result.mp4',
       '--out_height', str(OUT_HEIGHT), '--nosmooth','--pads', *map(str, PADS)]
if BOX:  cmd += ['--box',  *map(str, BOX)]
if CROP: cmd += ['--crop', *map(str, CROP)]
print('running:', ' '.join(cmd))
import subprocess
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-2500:])
if p.returncode != 0:
    print('--- ERROR ---'); print(p.stderr[-2500:])
else:
    print('OK -> /content/result.mp4')


## Step 5 - GFPGAN face restoration (sharpen the face)
Detects the face in each frame and restores realistic detail, then pastes it
back into the full frame. This is what fixes a soft/low-res face. Set
`ENHANCE = False` to skip. `GFPGAN_UPSCALE = 2` also enlarges the frame.


In [ ]:
ENHANCE = True
GFPGAN_UPSCALE = 1     # 1 = keep size (just restore); 2 = also 2x upscale
WORKING = '/content/result.mp4'
if ENHANCE:
    # install GFPGAN + fix the known torchvision.functional_tensor breakage
    !pip install -q gfpgan realesrgan >/dev/null 2>&1
    import glob
    for f in glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py'):
        s=open(f).read().replace('torchvision.transforms.functional_tensor','torchvision.transforms.functional')
        open(f,'w').write(s)
    !wget -q -c https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -O /content/GFPGANv1.4.pth
    import cv2
    from gfpgan import GFPGANer
    rest = GFPGANer(model_path='/content/GFPGANv1.4.pth', upscale=GFPGAN_UPSCALE, arch='clean', channel_multiplier=2, bg_upsampler=None)
    cap = cv2.VideoCapture('/content/result.mp4')
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames=[]
    n=0
    while True:
        ok, fr = cap.read()
        if not ok: break
        _,_,out = rest.enhance(fr, has_aligned=False, only_center_face=False, paste_back=True)
        frames.append(out); n+=1
    cap.release()
    print('enhanced', n, 'frames')
    h,w = frames[0].shape[:2]
    vw = cv2.VideoWriter('/content/gfpgan_silent.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w,h))
    for fr in frames: vw.write(fr)
    vw.release()
    # mux the synced audio (from result.mp4) back onto the enhanced video
    import os
    os.system('ffmpeg -y -loglevel error -i /content/gfpgan_silent.mp4 -i /content/result.mp4 -map 0:v -map 1:a -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest /content/result_enhanced.mp4')
    WORKING = '/content/result_enhanced.mp4'
    print('GFPGAN done ->', WORKING)
else:
    print('GFPGAN skipped ->', WORKING)


## Step 6 - (optional) resize to a target width, then preview + download


In [ ]:
TARGET_W = 1280   # 0 = keep as-is
import os
FINAL = WORKING
if TARGET_W:
    FINAL = '/content/final.mp4'
    os.system(f'ffmpeg -y -loglevel error -i "{WORKING}" -vf "scale={TARGET_W}:-2:flags=lanczos" -c:v libx264 -pix_fmt yuv420p -c:a aac "{FINAL}"')
    print('resized to width', TARGET_W, '->', FINAL)
print('final:', FINAL)


In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open(FINAL,'rb').read()).decode()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')


In [ ]:
from google.colab import files
files.download(FINAL)  # saves to Downloads; move it to D:\\MatrixVideos


---
### Notes
- If GFPGAN still isn't sharp enough, raise `OUT_HEIGHT` to 2112 (bigger face
  going in) and `GFPGAN_UPSCALE` to 2.
- Hard limit: GFPGAN hallucinates plausible detail but cannot recover a face
  that is only a handful of pixels. For the crispest talking face, the
  character should be closer to camera in the source clip.
- The whole parking-lot frame is preserved throughout; only the mouth is
  synced and the face restored.
